# Tracks Through a Predefined Area

Filter and visualize ship tracks that cross a geographic zone during a chosen time period.  
See [tuto.md](../tuto.md) for detailed explanations of every function and parameter.

## 1 — Setup

In [1]:
import sys, os
project_path = r"C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder"
if project_path not in sys.path:
    sys.path.append(project_path)

import track_builder as tb
import pandas as pd
import plotly.graph_objects as go

BASE_PATH      = r"C:\Users\lamin\Documents\maitrise\ASTD\data"
YEAR           = 2019
MONTHS_TO_LOAD = [1, 2, 3]
USECOLS        = "default"
COLS_REQUIRED  = ["shipid", "date_time_utc", "latitude", "longitude",
                  "astd_cat", "flagname", "iceclass", "sizegroup_gt"]

## 2 — Load data

In [2]:
# Sampled (first + last day) — for track building
df_for_track = tb.load_astd_monthly(
    BASE_PATH, YEAR, months=MONTHS_TO_LOAD, progress=True,
    usecols=USECOLS, sampling=[0, -1], remove_nan_rows=COLS_REQUIRED
)

# Full data — for visualization
df = tb.load_astd_monthly(
    BASE_PATH, YEAR, months=MONTHS_TO_LOAD, progress=True,
    usecols=USECOLS, sampling=None, remove_nan_rows=COLS_REQUIRED
)

c:\Users\lamin\miniconda3\envs\astd-track\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading ASTD CSVs: 100%|██████████| 3/3 [03:16<00:00, 65.37s/it]


## 3 — Build tracks

In [3]:
tracks = tb.build_ship_tracks(df_for_track, matching_strategy="balanced")

Data after cleaning:
  Date range: 2019-01-01 00:00:02+00:00 to 2019-03-31 23:59:57+00:00
  Ship types: ['offshore supply ships' 'other service offshore vessels'
 'general cargo ships' 'fishing vessels' 'passenger ships'
 'other activities' 'bulk carriers' 'ro-ro cargo ships' 'cruise ships'
 'refrigerated cargo ships' 'chemical tankers' 'container ships'
 'gas tankers' 'crude oil tankers' 'oil product tankers']
  Unique ships: 2618


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:230: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['period_month'] = df['date_time_utc'].dt.to_period('M')


Creating segments for 3059 unique ship-months (segments)...
Successfully created 3059 monthly segments.
Sample segment: fishing vessels|iceland|fs ice class 1c|< 1000 gt in 2019-02


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:382: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = (ship_means.groupby('astd_cat')


## 4 — Define an area of interest

Three options — pick one (see `tuto.md` for details):
- **region name** → `region="norway"`
- **Shapely geometry** → `bounding_box=box(...)`
- **GeoJSON / Shapefile path** → `bounding_box="path/to/file.geojson"`

In [4]:
from shapely.geometry import box

# Barents Sea zone: lon 15°–45°E, lat 68°–76°N
barents_zone = box(15.0, 68.0, 45.0, 76.0)

## 5 — Filter tracks crossing the zone

In [5]:
work = tb.build_light_multi_track_data(
    track_table=tracks,
    track_sampling=20,
    positions_df=df,
    n_tracks_length=2,
    bounding_box=barents_zone, # <----
)


computing typical speeds...


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:382: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = (ship_means.groupby('astd_cat')
C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:170: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_bwd = valid_fwd.shift(1).fillna(False) & is_same_ship_prev


Cleaning completed: 3552 'ghost' or aberrant points removed.
Using provided Shapely Geometry for spatial filter.
  -> Spatial Filter: Keeping 8 tracks out of 20.


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\io\astd_loader.py:647: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.iloc[::point_stride])


## 6 — Visualize

In [9]:
fig = tb.plot_ship_tracks(
    work, color_by="track_id", show_start_end=True,
    map_style="open-street-map",
    title="Tracks Through the Barents Sea Zone",
)

fig.show()

## 7 — Add a time filter

Use `date_from` / `date_to` to restrict the displayed period.

In [10]:
fig_time = tb.plot_ship_tracks(
    work, color_by="track_id", show_start_end=True,
    map_style="open-street-map",
    title="Barents Zone — Jan to Feb 2019",
    date_from="2019-01-25",
    date_to="2019-02-10",
)
fig_time.show()